In [2]:
import os
import sys
import pandas as pd
import json
import pickle

In [3]:
input_file = "join_est_record_jobjoin.txt"
output_file = "subquery_plans_jobjoin.json"

In [4]:
# get lines from input file
with open(input_file, "r") as f:
    lines = f.readlines()

dict_subqueries = {}

query_counter = -1
subquery_counter = 0

for line in lines:
    
    if line.find("query: 0") != -1:
        query_counter += 1
        subquery_counter = 0
        dict_subqueries[query_counter] = {}
        dict_subqueries[query_counter][subquery_counter] = {}
    elif line.find("query:") != -1:
        subquery_counter += 1
        dict_subqueries[query_counter][subquery_counter] = {}
        
    if line.find("inner_rel") != -1:
        inner_rel_ = True
    if line.find("outer_rel") != -1:
        inner_rel_ = False

    if line.find("RELOPTINFO") != -1:
        # get what is between parenthesis
        relname = line.split("(")[1].split(")")[0]
        
        if inner_rel_:
            dict_subqueries[query_counter][subquery_counter]["inner_rel"] = relname
        else:
            dict_subqueries[query_counter][subquery_counter]["outer_rel"] = relname
            
            

In [5]:
# dump in subquery_plans_jobjoin.json
with open(output_file, "w") as f:
    json.dump(dict_subqueries, f, indent=4)

In [6]:
# load subquery_plans_joblight.json
dict_subqueries = {}
with open(output_file) as f:
    dict_subqueries = json.load(f)

In [7]:
# read lines of /home/user/mayer/FactorJoin/imdb/jobjoin/job_join_queries.sql
with open("/home/user/mayer/FactorJoin/imdb/jobjoin/job_join_queries.sql", "r") as f:
    queries = f.readlines()

In [8]:
# read queries from files
def read_queries(queries, subqueries):
    dict_queries = {}
    dict_subqueries = {}
    query_counter = 0
    
    for q in queries:
        #q = q + ";"
        dict_queries[str(query_counter)] = q
        
        subqueries_q = subqueries[str(query_counter)]
        
        dict_subqueries[str(query_counter)] = []
        # iteratate over values in subqueries_q
        for q in subqueries_q.values():
            outer_ = q['outer_rel']
            outer_ = outer_.split(" ")
            # sorrt outer_ to get the correct order
            outer_ = sorted(outer_)
            outer_ = " ".join(outer_)
            tuple_ = (q['inner_rel'], outer_)
            dict_subqueries[str(query_counter)].append(tuple_)
        
        query_counter += 1
    
    return (dict_queries, dict_subqueries)



In [9]:
dict_qs = read_queries(queries, dict_subqueries)


In [10]:
# count number of subqueries
counter = 0
for q in dict_qs[1]:
    counter += len(dict_qs[1][q])
    
print("Number of subqueries: ", counter)

Number of subqueries:  23275


In [11]:
dict_qs[1]

{'0': [('mc', 'ct'),
  ('mi_idx', 'it'),
  ('mi_idx', 'mc'),
  ('t', 'mc'),
  ('t', 'mi_idx'),
  ('mi_idx', 'ct mc'),
  ('t', 'ct mc'),
  ('mc', 'it mi_idx'),
  ('t', 'it mi_idx'),
  ('t', 'mc mi_idx'),
  ('it', 'ct mc mi_idx'),
  ('t', 'ct mc mi_idx'),
  ('t', 'it mc mi_idx'),
  ('t', 'ct it mc mi_idx')],
 '1': [('mc', 'cn'),
  ('mk', 'k'),
  ('mk', 'mc'),
  ('t', 'mc'),
  ('t', 'mk'),
  ('mk', 'cn mc'),
  ('t', 'cn mc'),
  ('mc', 'k mk'),
  ('t', 'k mk'),
  ('t', 'mc mk'),
  ('k', 'cn mc mk'),
  ('t', 'cn mc mk'),
  ('t', 'k mc mk'),
  ('t', 'cn k mc mk')],
 '2': [('mk', 'k'),
  ('mk', 'mi'),
  ('t', 'mi'),
  ('t', 'mk'),
  ('mi', 'k mk'),
  ('t', 'k mk'),
  ('t', 'mi mk'),
  ('t', 'k mi mk')],
 '3': [('mi_idx', 'it'),
  ('mk', 'k'),
  ('mk', 'mi_idx'),
  ('t', 'mi_idx'),
  ('t', 'mk'),
  ('mk', 'it mi_idx'),
  ('t', 'it mi_idx'),
  ('mi_idx', 'k mk'),
  ('t', 'k mk'),
  ('t', 'mi_idx mk'),
  ('k', 'it mi_idx mk'),
  ('t', 'it mi_idx mk'),
  ('t', 'k mi_idx mk'),
  ('t', 'it k mi_idx

In [19]:
count_subs = 0
for subs in dict_qs[1]:
    if subs in ["28", "30"]:
        print("Excluding: ", subs)  
        continue
    elif subs in [28, 30]:
        print("Excluding: ", subs)  
        continue
    else:
        
        #print(len(dict_qs[1][subs]))
        count_subs += len(dict_qs[1][subs])
        
    
count_subs

Excluding:  28
Excluding:  30


9565

In [12]:
with open('dict_sql_imdb_jobjoin.pkl', 'wb') as f:
    pickle.dump(dict_qs, f)

In [33]:
dict_qs

({'0': 'SELECT COUNT(*) FROM company_type AS ct,      info_type AS it,      movie_companies AS mc,      movie_info_idx AS mi_idx,      title AS t WHERE ct.id = mc.company_type_id   AND t.id = mc.movie_id   AND t.id = mi_idx.movie_id   AND mc.movie_id = mi_idx.movie_id   AND it.id = mi_idx.info_type_id;\n',
  '1': 'SELECT COUNT(*) FROM company_name AS cn,      keyword AS k,      movie_companies AS mc,      movie_keyword AS mk,      title AS t WHERE cn.id = mc.company_id   AND mc.movie_id = t.id   AND t.id = mk.movie_id   AND mk.keyword_id = k.id   AND mc.movie_id = mk.movie_id;\n',
  '2': 'SELECT COUNT(*) FROM keyword AS k,      movie_info AS mi,      movie_keyword AS mk,      title AS t WHERE t.id = mi.movie_id   AND t.id = mk.movie_id   AND mk.movie_id = mi.movie_id   AND k.id = mk.keyword_id;\n',
  '3': 'SELECT COUNT(*) FROM info_type AS it,      keyword AS k,      movie_info_idx AS mi_idx,      movie_keyword AS mk,      title AS t WHERE t.id = mi_idx.movie_id   AND t.id = mk.movie_i